<a href="https://colab.research.google.com/github/nomanamir20/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nomanamir20/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# Week-7 setup: reproduce the validated Week-6 model and client-grouped scores

import os
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, average_precision_score


# ---------------------------------------------------------
# 1. Load the anonymized dataset
# ---------------------------------------------------------

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "nomanamir20/flyrank-ml-internship/main/"
    "data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_URL)

print("Dataset shape:", df.shape)


# Keep the same modeling lane as Week 5 / Week 6
df = df[df["content_type"] == "keyword article"].copy()

print("Keyword article lane shape:", df.shape)


# ---------------------------------------------------------
# 2. Recreate the Week-5 target
# ---------------------------------------------------------

df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

print("Overall decline rate:", round(
    df["is_declining_label"].mean(), 4
))


# ---------------------------------------------------------
# 3. Reproduce the Week-5 feature configuration
# ---------------------------------------------------------

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "impression_tier",
    "position_tier"
]

feature_columns = numeric_features + categorical_features

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Total model features:", len(feature_columns))


# ---------------------------------------------------------
# 4. Explicit leakage exclusions
# ---------------------------------------------------------

for forbidden in [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "client_id",
    "content_id"
]:
    print(
        forbidden,
        ":",
        "IN FEATURES" if forbidden in feature_columns else "excluded"
    )


# ---------------------------------------------------------
# 5. Reproduce the original grouped train/test split
# ---------------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        df,
        y=df["is_declining_label"],
        groups=df["client_id"]
    )
)

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()

X_train = train[feature_columns].copy()
X_test = test[feature_columns].copy()

y_train = train["is_declining_label"]
y_test = test["is_declining_label"]


# ---------------------------------------------------------
# 6. Recreate the Week-5 Logistic Regression
# ---------------------------------------------------------

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

model.fit(X_train, y_train)


# ---------------------------------------------------------
# 7. Independent client-grouped validation
# ---------------------------------------------------------

groups = df.loc[X_train.index, "client_id"]

group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

group_train_idx, group_valid_idx = next(
    group_splitter.split(
        X_train,
        y_train,
        groups=groups
    )
)

X_group_train = X_train.iloc[group_train_idx]
X_group_valid = X_train.iloc[group_valid_idx]

y_group_train = y_train.iloc[group_train_idx]
y_group_valid = y_train.iloc[group_valid_idx]

groups_train = groups.iloc[group_train_idx]
groups_valid = groups.iloc[group_valid_idx]

print("\nGrouped train shape:", X_group_train.shape)
print("Grouped validation shape:", X_group_valid.shape)
print("Training clients:", groups_train.nunique())
print("Validation clients:", groups_valid.nunique())

client_overlap = (
    set(groups_train.unique())
    & set(groups_valid.unique())
)

print("Client overlap:", len(client_overlap))


# ---------------------------------------------------------
# 8. Fit the validated grouped model
# ---------------------------------------------------------

grouped_model = clone(model)

grouped_model.fit(
    X_group_train,
    y_group_train
)

grouped_scores = grouped_model.predict_proba(
    X_group_valid
)[:, 1]


# ---------------------------------------------------------
# 9. Validation metrics
# ---------------------------------------------------------

def precision_at_k(y_true, scores, k):
    ranking = pd.DataFrame({
        "y_true": np.asarray(y_true),
        "score": np.asarray(scores)
    })

    ranking = ranking.sort_values(
        "score",
        ascending=False
    )

    return float(
        ranking.head(min(k, len(ranking)))["y_true"].mean()
    )


grouped_metrics = {
    "Precision@20": precision_at_k(
        y_group_valid,
        grouped_scores,
        20
    ),
    "Precision@50": precision_at_k(
        y_group_valid,
        grouped_scores,
        50
    ),
    "Precision@100": precision_at_k(
        y_group_valid,
        grouped_scores,
        100
    ),
    "ROC-AUC": roc_auc_score(
        y_group_valid,
        grouped_scores
    ),
    "Average Precision": average_precision_score(
        y_group_valid,
        grouped_scores
    )
}

print("\nValidated grouped metrics:")

for metric, value in grouped_metrics.items():
    print(f"{metric}: {value:.4f}")


print(
    "\nValidation positive rate:",
    f"{y_group_valid.mean():.4f}"
)

print(
    "Validation positive rate:",
    f"{y_group_valid.mean() * 100:.2f}%"
)

Dataset shape: (30000, 44)
Keyword article lane shape: (27207, 44)
Overall decline rate: 0.561
Numeric features: 22
Categorical features: 8
Total model features: 30
trend_direction : excluded
trend_pct : excluded
is_declining_label : excluded
client_id : excluded
content_id : excluded

Grouped train shape: (18636, 30)
Grouped validation shape: (2789, 30)
Training clients: 19
Validation clients: 5
Client overlap: 0

Validated grouped metrics:
Precision@20: 0.9500
Precision@50: 0.9600
Precision@100: 0.9600
ROC-AUC: 0.6696
Average Precision: 0.8568

Validation positive rate: 0.7418
Validation positive rate: 74.18%


## 1. Ranked actions + reason codes

The validated model is used as a decision-support ranking tool. Its purpose is to help a content team decide which keyword-article rows should be reviewed first.

The strongest validation evidence comes from a grouped client holdout with no client overlap between training and validation. The validation set contains 2,789 rows across 5 held-out clients.

The model achieved Precision@20 of 0.95, Precision@50 of 0.96, and Precision@100 of 0.96. ROC-AUC was 0.6696 and Average Precision was 0.8568. These results support using the ranking as a directional prioritization tool, not as an automatic decision system.

### Priority logic

- **HIGH_DECLINE_PRIORITY** — highest-ranked candidates that should be reviewed first.
- **MEDIUM_DECLINE_PRIORITY** — candidates that should be reviewed after the highest-priority queue.
- **LOW_DECLINE_PRIORITY** — lower-ranked candidates that can be monitored or reviewed later.

### Reason codes

- **HIGH_DECLINE_PRIORITY** — the model gives the candidate a high relative decline score.
- **MEDIUM_DECLINE_PRIORITY** — the candidate has a meaningful but lower relative decline score.
- **LOW_DECLINE_PRIORITY** — the candidate has a lower relative decline score and is not an immediate review priority.

### Recommended action

The recommended action is **human review of the content and its recent performance context**. The model does not automatically rewrite, delete, publish, redirect, or otherwise change content.

The ranking answers a practical question: **which candidates should a content team inspect first?**

In [2]:
# ---------------------------------------------------------
# Build the ranked content action queue
# ---------------------------------------------------------

ranked_queue = X_group_valid.copy()

ranked_queue = ranked_queue.reset_index()

ranked_queue["decline_score"] = grouped_scores
ranked_queue["observed_decline_label"] = (
    y_group_valid.reset_index(drop=True)
)

# Rank highest model score first
ranked_queue = ranked_queue.sort_values(
    "decline_score",
    ascending=False
).reset_index(drop=True)

ranked_queue["rank"] = (
    np.arange(len(ranked_queue)) + 1
)


# ---------------------------------------------------------
# Assign practical priority bands
# ---------------------------------------------------------

def assign_priority(rank):
    if rank <= 20:
        return "HIGH_DECLINE_PRIORITY"
    elif rank <= 100:
        return "MEDIUM_DECLINE_PRIORITY"
    else:
        return "LOW_DECLINE_PRIORITY"


ranked_queue["reason_code"] = (
    ranked_queue["rank"]
    .apply(assign_priority)
)


# ---------------------------------------------------------
# Human-readable recommended action
# ---------------------------------------------------------

ranked_queue["recommended_action"] = (
    "Human review of content and recent performance context"
)


# ---------------------------------------------------------
# Display the first 20 candidates
# ---------------------------------------------------------

display_columns = [
    "rank",
    "decline_score",
    "reason_code",
    "recommended_action"
]

print("Top 20 ranked candidates:")

display(
    ranked_queue[display_columns].head(20)
)

print("\nTotal ranked candidates:", len(ranked_queue))

Top 20 ranked candidates:


,rank,decline_score,reason_code,recommended_action
0,1,0.844718,HIGH_DECLINE_PRIORITY,Human review of content and recent performance...
1,2,0.802756,HIGH_DECLINE_PRIORITY,Human review of content and recent performance...
2,3,0.795945,HIGH_DECLINE_PRIORITY,Human review of content and recent performance...
3,4,0.793694,HIGH_DECLINE_PRIORITY,Human review of content and recent performance...
4,5,0.786352,HIGH_DECLINE_PRIORITY,Human review of content and recent performance...
5,6,0.780980,HIGH_DECLINE_PRIORITY,Human review of content and recent performance...
6,7,0.776114,HIGH_DECLINE_PRIORITY,Human review of content and recent performance...
7,8,0.772647,HIGH_DECLINE_PRIORITY,Human review of content and recent performance...
8,9,0.772457,HIGH_DECLINE_PRIORITY,Human review of content and recent performance...
9,10,0.769655,HIGH_DECLINE_PRIORITY,Human review of content and recent performance...



Total ranked candidates: 2789


## 2. Intended use and limits

### Intended use

This playbook is intended for content teams, SEO researchers, or analysts who need to prioritize keyword-article pages for human review.

The model ranks candidates according to their estimated likelihood of being in the declining class. The ranked queue can help a reviewer decide which pages to inspect first when time is limited.

The output is decision-support, not an automatic content management system.

### What the model can support

The model can support:

- Prioritizing pages for review.
- Identifying a small high-priority review queue.
- Comparing candidates by relative model score.
- Helping analysts focus limited review time on the highest-ranked candidates.
- Supporting content refresh planning after a human reviews the underlying page and context.

The grouped client validation provides evidence that the ranking can separate higher-priority candidates reasonably well on held-out clients. Precision@20 was 0.95, Precision@50 was 0.96, and Precision@100 was 0.96.

### Limits

The model does not prove that a page will decline in the future.

The model score is not a guaranteed probability of future traffic loss. The ranking should be interpreted as directional evidence.

The validation was performed on held-out clients from the available dataset. Results may change when the content mix, clients, search environment, or measurement period changes.

The model also does not explain the exact cause of a decline. A high-ranked page could require investigation of content freshness, search intent, competition, technical issues, seasonality, or other factors.

The playbook is therefore intended for research and decision support rather than production automation.

### Important boundary

A human must review the page and its context before any content change is made.

In [3]:
# ---------------------------------------------------------
# Section 2: Intended-use validation evidence
# ---------------------------------------------------------

print("INTENDED USE")
print(
    "Decision-support ranking for prioritizing keyword-article "
    "rows for human review."
)

print("\nVALIDATION EVIDENCE")

for metric, value in grouped_metrics.items():
    print(f"{metric}: {value:.4f}")

print("\nValidation rows:", len(X_group_valid))
print("Validation clients:", groups_valid.nunique())
print("Client overlap:", len(client_overlap))

print(
    "\nBoundary:",
    "Scores prioritize review; they do not automatically trigger "
    "content changes."
)

INTENDED USE
Decision-support ranking for prioritizing keyword-article rows for human review.

VALIDATION EVIDENCE
Precision@20: 0.9500
Precision@50: 0.9600
Precision@100: 0.9600
ROC-AUC: 0.6696
Average Precision: 0.8568

Validation rows: 2789
Validation clients: 5
Client overlap: 0

Boundary: Scores prioritize review; they do not automatically trigger content changes.


## 3. Human review + the no-go list

### Human review rules

Every high-priority recommendation must be reviewed by a person before any content action is taken.

The reviewer should check:

1. Whether the page is actually declining or showing a meaningful negative trend.
2. Whether the page has enough recent data to justify a decision.
3. Whether the current search intent still matches the page.
4. Whether the content is outdated or missing important information.
5. Whether competitors or search results have changed.
6. Whether the decline could be explained by seasonality or another external factor.
7. Whether the recommended refresh would improve the page without damaging useful existing content.

The model score is only a prioritization signal. A high score does not mean that a page definitely needs a content rewrite.

### What should NOT be automated

The following actions should not be automatically performed by this playbook:

- Publishing content changes.
- Rewriting an article automatically.
- Deleting content.
- Redirecting URLs.
- Changing search intent automatically.
- Changing important SEO metadata without review.
- Removing useful content based only on the model score.
- Making business-critical decisions from the ranking alone.
- Treating the model score as proof of causation.
- Sending recommendations directly to production without human approval.

### Human decision rule

The recommended workflow is:

**Model ranking → human inspection → contextual checks → decision → manual action**

The model helps decide where to look first. A human remains responsible for deciding what action, if any, should be taken.

In [4]:
# ---------------------------------------------------------
# Section 3: Human review and no-go checks
# ---------------------------------------------------------

review_rules = [
    "Check whether the observed trend supports the model signal.",
    "Check that enough recent data is available.",
    "Check current search intent and SERP context.",
    "Check content freshness and missing information.",
    "Check competitors and major search-result changes.",
    "Check seasonality or other external explanations.",
    "Confirm that the proposed change is unlikely to remove useful content."
]

no_go_cases = [
    "Automatic publishing",
    "Automatic article rewriting",
    "Automatic content deletion",
    "Automatic URL redirects",
    "Automatic search-intent changes",
    "Automatic SEO metadata changes",
    "Business-critical decisions from model score alone",
    "Treating model score as causal evidence",
    "Sending changes directly to production without review"
]

print("HUMAN REVIEW RULES")
for i, rule in enumerate(review_rules, start=1):
    print(f"{i}. {rule}")

print("\nNO-GO CASES")
for i, case in enumerate(no_go_cases, start=1):
    print(f"{i}. {case}")

print("\nDecision workflow:")
print("Model ranking -> human inspection -> contextual checks -> decision -> manual action")

HUMAN REVIEW RULES
1. Check whether the observed trend supports the model signal.
2. Check that enough recent data is available.
3. Check current search intent and SERP context.
4. Check content freshness and missing information.
5. Check competitors and major search-result changes.
6. Check seasonality or other external explanations.
7. Confirm that the proposed change is unlikely to remove useful content.

NO-GO CASES
1. Automatic publishing
2. Automatic article rewriting
3. Automatic content deletion
4. Automatic URL redirects
5. Automatic search-intent changes
6. Automatic SEO metadata changes
7. Business-critical decisions from model score alone
8. Treating model score as causal evidence
9. Sending changes directly to production without review

Decision workflow:
Model ranking -> human inspection -> contextual checks -> decision -> manual action


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.